# Project: NLP for Oil & Gas Industry

## Project title:
AI-Based Analysis of Incident Reports for Risk Detection

## Problem statement

In the Oil & Gas industry, daily operations generate large volumes of unstructured text such as:
- Incident reports
- Maintenance logs
- Safety observations

These reports are manually reviewed, which leads to:
- Delays in identifying high-risk incidents
- Inconsistent risk classification
- Missed patterns that could prevent accidents

## Project goal

The goal of this project is to build an NLP-based system that can:

1. Automatically analyze incident reports
2. Classify the risk level (High, Medium, Low)
3. Extract important entities (equipment, location, etc.)
4. Provide insights to support faster and better decision-making

## Expected outcome

A working prototype that:
- Takes incident text as input
- Outputs:
  - Risk level
  - Sentiment score
  - Key entities

This system can assist safety teams in prioritizing critical incidents.

## Tools:
- NLTK
- spaCy
- Scikit-learn
- Pandas

In [ ]:
#Install Libraries
# !pip install nltk spacy scikit-learn pandas

In [2]:
#Import Libraries
import pandas as pd
import nltk
import spacy

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('vader_lexicon')

nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\M.faisal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\M.faisal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\M.faisal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\M.faisal\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


# DATASET

In [3]:
#Sample Dataset
data = {
    "Report": [
        "Gas leak detected near pipeline section A",
        "Routine inspection completed successfully",
        "Minor oil spill contained quickly",
        "Equipment failure caused pressure surge",
        "Maintenance completed without issues",
        "Fire outbreak in offshore platform",
        "Valve malfunction detected during operation",
        "System operating under normal conditions"
    ],
    "Risk": [
        "High", "Low", "Medium", "High", "Low", "High", "Medium", "Low"
    ]
}

df = pd.DataFrame(data)
df

,Report,Risk
0,Gas leak detected near pipeline section A,High
1,Routine inspection completed successfully,Low
2,Minor oil spill contained quickly,Medium
3,Equipment failure caused pressure surge,High
4,Maintenance completed without issues,Low
5,Fire outbreak in offshore platform,High
6,Valve malfunction detected during operation,Medium
7,System operating under normal conditions,Low


# TEXT PREPROCESSING

### Text preprocessing steps:
- Tokenization
- Lowercasing
- Stopword removal
- Lemmatization

In [4]:
#Preprocessing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess(text):
    tokens = word_tokenize(text)
    tokens = [word.lower() for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stop_words]
    
    doc = nlp(" ".join(tokens))
    tokens = [token.lemma_ for token in doc]
    
    return " ".join(tokens)

df["Cleaned"] = df["Report"].apply(preprocess)
df

,Report,Risk,Cleaned
0,Gas leak detected near pipeline section A,High,gas leak detect near pipeline section
1,Routine inspection completed successfully,Low,routine inspection complete successfully
2,Minor oil spill contained quickly,Medium,minor oil spill contain quickly
3,Equipment failure caused pressure surge,High,equipment failure cause pressure surge
4,Maintenance completed without issues,Low,maintenance complete without issue
5,Fire outbreak in offshore platform,High,fire outbreak offshore platform
6,Valve malfunction detected during operation,Medium,valve malfunction detect operation
7,System operating under normal conditions,Low,system operate normal condition


# SENTIMENT ANALYSIS

**Sentiment Analysis** is used to detect severity tone of incident reports.

In [5]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

df["Sentiment"] = df["Report"].apply(lambda x: sia.polarity_scores(x)['compound'])
df

,Report,Risk,Cleaned,Sentiment
0,Gas leak detected near pipeline section A,High,gas leak detect near pipeline section,-0.3400
1,Routine inspection completed successfully,Low,routine inspection complete successfully,0.4939
2,Minor oil spill contained quickly,Medium,minor oil spill contain quickly,0.0000
3,Equipment failure caused pressure surge,High,equipment failure cause pressure surge,-0.6705
4,Maintenance completed without issues,Low,maintenance complete without issue,0.0000
5,Fire outbreak in offshore platform,High,fire outbreak offshore platform,-0.3400
6,Valve malfunction detected during operation,Medium,valve malfunction detect operation,0.0000
7,System operating under normal conditions,Low,system operate normal condition,0.0000


# NAMED ENTITY RECOGNITION (NER)

Named Entity Recognition (NER) is used to extract key entities such as:
- Locations
- Equipment

In [6]:
def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

df["Entities"] = df["Report"].apply(extract_entities)
df

,Report,Risk,Cleaned,Sentiment,Entities
0,Gas leak detected near pipeline section A,High,gas leak detect near pipeline section,-0.3400,[]
1,Routine inspection completed successfully,Low,routine inspection complete successfully,0.4939,[]
2,Minor oil spill contained quickly,Medium,minor oil spill contain quickly,0.0000,[]
3,Equipment failure caused pressure surge,High,equipment failure cause pressure surge,-0.6705,[]
4,Maintenance completed without issues,Low,maintenance complete without issue,0.0000,[]
5,Fire outbreak in offshore platform,High,fire outbreak offshore platform,-0.3400,[]
6,Valve malfunction detected during operation,Medium,valve malfunction detect operation,0.0000,[]
7,System operating under normal conditions,Low,system operate normal condition,0.0000,[]


# TEXT CLASSIFICATION (RISK PREDICTION)

### Risk Classification Model

We will train a model to predict:
- High / Medium / Low risk

## Feature Extraction

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df["Cleaned"])
y = df["Risk"]

## Train Model

We chose **Multinomial Naive Bayes** because it is simple, fast, and works very well with text data represented as word counts. It is a good starting point before moving to more complex models.

**Note**: In Machine Learning, the best model is not the most complex one, it’s the one that fits the data and solves the problem efficiently.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


## Evaluate Model

In [9]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.5


## Communication of results

After building and evaluating the model, it is important to communicate the findings clearly to stakeholders.

### Model performance
- The model was trained to classify incident reports into risk levels.
- Accuracy score indicates how well the model performs on unseen data.

### Key insights
- Reports containing words like "leak", "fire", "failure" are often classified as High risk.
- Routine and maintenance-related reports are typically Low risk.
- Medium risk cases often involve contained or minor issues.

### Business impact
- Faster identification of high-risk incidents
- Reduced manual effort in reviewing reports
- Improved safety monitoring and decision-making

### Limitations
- Small dataset may affect accuracy
- Model may misclassify ambiguous reports
- Requires more real-world data for improvement

### Recommendations
- Collect more incident data for training
- Improve preprocessing and feature engineering
- Test additional models (e.g., Logistic Regression)

### Conclusion
This NLP system demonstrates how text data can be transformed into actionable insights in the Oil & Gas industry.

### LET'S TEST THE MODEL WITH NEW INPUT:

In [10]:
new_text = ["Gas leakage reported in storage tank"]

cleaned = [preprocess(t) for t in new_text]
vector = vectorizer.transform(cleaned)

prediction = model.predict(vector)

print("Predicted Risk Level:", prediction[0])

Predicted Risk Level: High


## Real-time usage scenario

Example:

Input:
"Gas leakage reported in storage tank"

System Output:
- Risk Level: High
- Sentiment: Negative
- Entities: Gas leakage, storage tank

### Interpretation:
This allows safety officers to immediately prioritize this incident.

## Visual inspection

In [11]:
# NER Visualization
from spacy import displacy

doc = nlp("Fire outbreak in offshore platform")

displacy.render(doc, style="ent", jupyter=True)

C:\Users\M.faisal\Desktop\ai_ml\env\lib\site-packages\spacy\displacy\__init__.py:214: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


### Summary:

In this project, we:
- Processed incident report text
- Performed sentiment analysis
- Extracted entities using NER
- Built a machine learning model to classify risk

## Real-World Impact:
- Faster incident analysis
- Improved safety monitoring
- Early detection of high-risk events

This is how NLP can be applied in the Oil & Gas industry.

## Save the Model + Vectorizer

In [16]:
import joblib

joblib.dump({
    "model": model,
    "vectorizer": vectorizer
}, "nlp_pipeline.pkl")

print("Model and vectorizer saved successfully.")

Model and vectorizer saved successfully.


**Why save both?:**

- The model expects input in the same transformed format, without the vectorizer, predictions will break.

To load the model?

In [17]:
import joblib

data = joblib.load("nlp_pipeline.pkl")

model = data["model"]
vectorizer = data["vectorizer"]

print("Model loaded successfully.")

Model loaded successfully.


In [18]:
# To use the saved model for prediction
def predict_risk(text):
    cleaned = preprocess(text)
    vector = vectorizer.transform([cleaned])
    prediction = model.predict(vector)
    return prediction[0]

# Test
print(predict_risk("Gas leak detected in pipeline"))

High


Note: Your model is useless without the exact preprocessing pipeline.

## CHALLENGE?

With the simple project understanding which you have gained so far, use it to build something like this:- (https://emotrack.streamlit.app/)

Github Repo: https://github.com/devmab24/emotion_detector